# Day 16 — Spark UI & Troubleshooting

**What you will learn:**
- Navigating the Spark UI: Jobs, Stages, Tasks, SQL, Storage tabs
- Reading `explain()` output: logical vs physical plans
- Identifying skew, spill, and stragglers from stage metrics
- Common performance problems and how to diagnose them
- Key Spark metrics: shuffle read/write bytes, spill to disk, GC time

**Exam relevance:** Architecture (20%) + Optimisation (10%) — Spark UI interpretation, reading query plans, and diagnosing shuffle problems are exam topics.

In [ ]:
from pyspark.sql.functions import col, count, avg, sum, round, broadcast, expr
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

## 1. The Spark UI — Tab by Tab

```
http://localhost:4040  (driver host)
```

| Tab | What you see | When to look |
|---|---|---|
| **Jobs** | All actions triggered, their stages, duration | First stop — identify slow jobs |
| **Stages** | Task distribution, shuffle metrics, skew | Diagnose slow stages |
| **Tasks** | Per-task input/output/GC breakdown | Identify straggler tasks |
| **SQL** | DAG of SQL/DataFrame operations | Read physical plan visually |
| **Storage** | Cached RDDs/DataFrames, fraction cached | Verify cache is being used |
| **Environment** | Spark config, JVM flags | Confirm settings applied |
| **Executors** | Heap, GC time, shuffle per executor | Spot uneven load |

## 2. Reading explain() Plans

`explain(mode)` supports:
- `"simple"` — just the physical plan
- `"extended"` — logical + optimised logical + physical
- `"codegen"` — generated Java code
- `"cost"` — with cost estimates
- `"formatted"` — formatted with node IDs (most readable)

In [ ]:
data = [
    ("Alice", "Engineering", 95000),
    ("Bob",   "Marketing",   72000),
    ("Carol", "Engineering", 110000),
    ("Dave",  "Sales",       58000),
    ("Eve",   "Marketing",   81000),
]
df = spark.createDataFrame(data, ["name", "dept", "salary"])

query = df.filter(col("salary") > 70000) \
          .groupBy("dept") \
          .agg(round(avg("salary"), 0).alias("avg_salary"))

# Simple physical plan
print("=== SIMPLE ===")
query.explain("simple")

# Extended — shows all plan stages
print("\n=== EXTENDED ===")
query.explain("extended")

## 3. Key Physical Plan Nodes

| Node | Meaning | Performance implication |
|---|---|---|
| `FileScan` | Reading from storage | Check pushed filters |
| `Filter` | Predicate applied | If above FileScan, not pushed down |
| `Project` | Column selection | Cheap |
| `HashAggregate` | Local partial agg | Two-phase: partial + final |
| `Exchange` | **Shuffle** | Expensive — network I/O |
| `Sort` | Sort before join/agg | Requires partition to be sorted |
| `SortMergeJoin` | Join via sort + merge | Both sides shuffled |
| `BroadcastHashJoin` | Small side broadcast | No shuffle on small side |
| `InMemoryTableScan` | Reading from cache | Cache is being used |
| `AdaptiveSparkPlan` | AQE wrapper | Plan may change at runtime |

In [ ]:
# Demonstrate broadcast join in the plan
dept_info = spark.createDataFrame([
    ("Engineering", "Tech"),
    ("Marketing",   "Ops"),
    ("Sales",       "Ops"),
], ["dept", "division"])

print("=== SortMergeJoin (no broadcast) ===")
df.join(dept_info, on="dept").explain("simple")

print("\n=== BroadcastHashJoin ===")
df.join(broadcast(dept_info), on="dept").explain("simple")

## 4. Diagnosing Data Skew

**Skew** = one partition has far more data than others → one task takes 10× longer.

**How to detect in Spark UI:**
- Stages tab → look at task distribution: max >> median duration
- Tasks tab → one task has huge input bytes or shuffle read

**Common causes:**
- Joining on a column with one dominant value (e.g., `null`, `"Unknown"`)
- GroupBy on low-cardinality key with one huge group

In [ ]:
# Simulate skewed data — one key dominates
import random
random.seed(42)

skewed = [
    ("hot_key" if random.random() < 0.9 else f"key_{i}", random.randint(1, 1000))
    for i in range(10000)
]
df_skew = spark.createDataFrame(skewed, ["key", "value"])

# Check distribution — skew visible here
df_skew.groupBy("key").count() \
    .orderBy(col("count").desc()) \
    .show(5)

# Fix option 1: salting — add random suffix to hot key, then aggregate in two passes
from pyspark.sql.functions import concat, lit, floor, rand

NUM_SALTS = 10
df_salted = df_skew.withColumn(
    "salted_key",
    when(col("key") == "hot_key",
         concat(col("key"), lit("_"), (floor(rand() * NUM_SALTS)).cast("string")))
    .otherwise(col("key"))
)

# First aggregation — distributed across salt buckets
partial = df_salted.groupBy("salted_key").agg(sum("value").alias("partial_sum"))

# Strip salt and aggregate again
from pyspark.sql.functions import regexp_replace
final = partial \
    .withColumn("key", regexp_replace(col("salted_key"), r"_\d+$", "")) \
    .groupBy("key").agg(sum("partial_sum").alias("total"))

final.orderBy(col("total").desc()).show(5)

## 5. Diagnosing Spill

**Spill** = data that doesn't fit in executor memory is written to disk during shuffle.

**How to detect:**
- Stage metrics → "Shuffle spill (memory)" and "Shuffle spill (disk)" > 0
- Executor metrics → heap usage near capacity

**Fixes:**
1. Increase executor memory (`spark.executor.memory`)
2. Increase parallelism (more partitions → smaller per-partition data)
3. Reduce data earlier (push filters up, project fewer columns)

In [ ]:
# Partition count control — coalesce vs repartition
df_big = spark.range(1_000_000).toDF("id")

print("Default partitions:", df_big.rdd.getNumPartitions())

# repartition — full shuffle, evenly distributed
print("After repartition(16):", df_big.repartition(16).rdd.getNumPartitions())

# coalesce — no shuffle, merges existing partitions (use to reduce only)
print("After coalesce(4):",     df_big.coalesce(4).rdd.getNumPartitions())

## 6. AQE — Adaptive Query Execution

AQE re-optimises the plan at runtime using actual partition statistics.

| Feature | What it does |
|---|---|
| Coalescing post-shuffle partitions | Merges small post-shuffle partitions (avoids thousands of tiny tasks) |
| Converting sort-merge to broadcast join | If a join side turns out small, switches to BroadcastHashJoin |
| Skew join optimisation | Splits skewed partitions into smaller sub-tasks |

AQE is **on by default** in Spark 3.x. Verify with:

In [ ]:
print("AQE enabled:",
      spark.conf.get("spark.sql.adaptive.enabled"))
print("Post-shuffle coalesce:",
      spark.conf.get("spark.sql.adaptive.coalescePartitions.enabled"))
print("Shuffle partitions:",
      spark.conf.get("spark.sql.shuffle.partitions"))

## 7. Common Performance Patterns

| Symptom | Likely cause | Fix |
|---|---|---|
| One task takes 10× longer | Data skew | Salt the key, AQE skew join |
| Many tiny tasks | Too many shuffle partitions | AQE coalesce, lower `shuffle.partitions` |
| High shuffle write/read | Unnecessary shuffles | Broadcast small tables, push filters |
| Disk spill > 0 | Executor memory too low | Increase memory or parallelism |
| High GC time (>10%) | Too many objects in heap | Tune memory fractions, use serialization |
| `SortMergeJoin` on small table | Broadcast threshold too low | Raise `autoBroadcastJoinThreshold` or use `broadcast()` |
| Filter after FileScan | Predicate not pushed | Check partition pruning / pushdown |

---

## Day 16 Checklist

- [ ] Know the 7 Spark UI tabs and what to look at in each
- [ ] Read a physical plan from `explain()` — identify Exchange (shuffle) nodes
- [ ] Confirmed `BroadcastHashJoin` vs `SortMergeJoin` in explain output
- [ ] Know how to detect data skew from Spark UI task duration distribution
- [ ] Know that AQE is on by default in Spark 3.x and what it auto-optimises
- [ ] Know the difference between `repartition` (full shuffle) and `coalesce` (no shuffle)

**Next:** Day 17 — Memory Management & Accumulators